# 二吸収帯整合型推定のための合成メタンプルーム注入・評価

HISUIの実観測背景スペクトルに既知のメタン濃度増分を注入し、1.6 µm帯と2.3 µm帯を用いた推定性能を評価する。

MODTRAN LUTの濃度軸は、**背景からの増分ではなく、大気中メタンの絶対濃度**とする。

背景大気中のメタン濃度を $c_{\mathrm{bg}}$、注入する濃度増分を $\Delta c$ とすると、注入後の絶対濃度は

$$
c_{\mathrm{total}}(x,y)=c_{\mathrm{bg}}+\Delta c(x,y)
$$

です。

合成注入には次の放射輝度比を使う。

$$
T(\lambda,\Delta c)=
\frac{L_{\mathrm{MODTRAN}}(\lambda,c_{\mathrm{bg}}+\Delta c)}
{L_{\mathrm{MODTRAN}}(\lambda,c_{\mathrm{bg}})}
$$

したがって、合成スペクトルは

$$
L_{\mathrm{syn}}(x,y,\lambda)=
L_{\mathrm{bg}}(x,y,\lambda)
T\left(\lambda,\Delta c_{\mathrm{true}}(x,y)\right)
$$

となる。

この比ではMODTRAN放射輝度の一定倍率が分子・分母で相殺されるため、MODTRANを100倍してHISUIと単位を合わせる処理は不要。

## このNotebookの位置づけ

- 注入前HISUI cubeを真の背景として使うOracle評価
- 1.6 µm帯単独の線形MF
- 2.3 µm帯単独の線形MF
- 両帯域に共通の濃度増分を課すLUT探索
- 非線形最小二乗による連続値精密化
- 真値とのRMSE、Bias、$R^2$、検出性能比較

完全なSC-LMMFに必要なスペクトルゲイン、傾き、波長シフトなどの同時推定は、まだ含めていない。

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import least_squares

np.set_printoptions(precision=5, suppress=True)


## 1. 設定

In [ ]:
ROI_CSV = r"E:\refit\all_roi_spectra.csv"
CH4_LUT_CSV = r"E:\refit\CH4b.csv"

# MODTRAN LUTの濃度軸は絶対濃度 [ppm]
BACKGROUND_CH4_PPM = 1.8
MAX_ENHANCEMENT_PPM = 0.7
ENHANCEMENT_STEP_PPM = 0.02

FWHM_NM = 12.5
WINDOW_16 = (1580.0, 1750.0)
WINDOW_23 = (2100.0, 2450.0)
UAS_MAX_ENHANCEMENT_PPM = 0.5

PLUME_PEAK_ENHANCEMENT_PPM = 0.6
PLUME_CENTER_YX = None
PLUME_ANGLE_DEG = 20.0
PLUME_DECAY_PIX = 18.0
PLUME_CROSS_SIGMA_PIX = 4.0
PLUME_SOURCE_SIGMA_PIX = 2.0

## 2. HISUI ROI スペクトルCSVの読み込み

In [ ]:
def get_wave_columns(df):
    pattern = re.compile(r"^wave_([0-9.]+)nm$")
    pairs = []
    for col in df.columns:
        match = pattern.match(str(col))
        if match is not None:
            pairs.append((col, float(match.group(1))))
    if not pairs:
        raise ValueError(f"wave_***nm形式の列が見つかりません。先頭列: {list(df.columns[:10])}")
    pairs.sort(key=lambda item: item[1])
    return [p[0] for p in pairs], np.array([p[1] for p in pairs], dtype=float)


def load_roi_spectra_csv(path):
    df = pd.read_csv(path)
    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("CSVには y と x 列が必要です。")
    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(df, spectra, fill_value=np.nan):
    frame = df.reset_index(drop=True)
    ys = np.sort(frame["y"].unique())
    xs = np.sort(frame["x"].unique())
    y_to_i = {v:i for i,v in enumerate(ys)}
    x_to_i = {v:i for i,v in enumerate(xs)}
    cube = np.full((len(ys), len(xs), spectra.shape[1]), fill_value, dtype=float)
    for row_i, row in frame.iterrows():
        cube[y_to_i[row["y"]], x_to_i[row["x"]]] = spectra[row_i]
    return cube, ys, xs


def make_valid_pixel_mask(cube, nodata_values=(0.0, -9999.0), require_positive=True):
    valid = np.isfinite(cube)
    for value in nodata_values:
        valid &= cube != value
    if require_positive:
        valid &= cube > 0
    return np.all(valid, axis=2)

In [ ]:
df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube_background_true, y_values, x_values = spectra_to_cube(df, spectra)
valid_mask = make_valid_pixel_mask(cube_background_true)

print("DataFrame shape:", df.shape)
print("Cube shape:", cube_background_true.shape)
print("Wavelength range:", wavelengths[0], "to", wavelengths[-1], "nm")
print("Valid pixels:", int(valid_mask.sum()), "/", valid_mask.size)

## 3. MODTRAN CH₄ LUTの読み込みと装置関数畳み込み

In [ ]:
def load_ch4_absolute_concentration_lut(path):
    df_lut = pd.read_csv(path)
    candidates = [c for c in df_lut.columns if str(c).strip().lower() in {"wavelength", "wave", "wavelength_nm"}]
    if not candidates:
        raise ValueError("LUTに wavelength 列が見つかりません。")
    wave_col = candidates[0]
    mod_wave = df_lut[wave_col].to_numpy(dtype=float)

    pairs = []
    for col in df_lut.columns:
        if col == wave_col:
            continue
        try:
            pairs.append((col, float(str(col).strip())))
        except ValueError:
            pass
    if len(pairs) < 2:
        raise ValueError("LUTに絶対メタン濃度を表す数値列が2列以上必要です。")
    pairs.sort(key=lambda item: item[1])
    abs_grid = np.array([p[1] for p in pairs], dtype=float)
    spectra = df_lut[[p[0] for p in pairs]].to_numpy(dtype=float).T
    order = np.argsort(mod_wave)
    return mod_wave[order], abs_grid, spectra[:, order]


def gaussian_srf_resample(mod_wave, mod_spectra, sensor_wave, fwhm_nm):
    mod_wave = np.asarray(mod_wave, float)
    mod_spectra = np.asarray(mod_spectra, float)
    sensor_wave = np.asarray(sensor_wave, float)
    fwhm = np.full(sensor_wave.shape, float(fwhm_nm)) if np.isscalar(fwhm_nm) else np.asarray(fwhm_nm, float)
    if fwhm.shape != sensor_wave.shape:
        raise ValueError("FWHM配列の長さがsensor_waveと一致しません。")

    out = np.full((mod_spectra.shape[0], sensor_wave.size), np.nan)
    for j, center in enumerate(sensor_wave):
        sigma = fwhm[j] / (2.0*np.sqrt(2.0*np.log(2.0)))
        use = np.abs(mod_wave-center) <= 4.0*sigma
        if use.sum() < 2:
            out[:, j] = np.array([np.interp(center, mod_wave, s) for s in mod_spectra])
        else:
            w = np.exp(-0.5*((mod_wave[use]-center)/sigma)**2)
            w /= w.sum()
            out[:, j] = mod_spectra[:, use] @ w
    return out

In [ ]:
modtran_wavelengths, absolute_concentration_grid, modtran_spectra = load_ch4_absolute_concentration_lut(CH4_LUT_CSV)
sensor_lut_absolute = gaussian_srf_resample(
    modtran_wavelengths, modtran_spectra, wavelengths, FWHM_NM
)

print("Absolute CH4 concentration grid [ppm]:", absolute_concentration_grid)
print("Sensor-resolution LUT shape:", sensor_lut_absolute.shape)

## 4. 背景濃度基準の濃度増分LUTを作成

In [ ]:
def interpolate_absolute_lut_spectrum(concentration_ppm, concentration_grid, sensor_lut):
    if not concentration_grid.min() <= concentration_ppm <= concentration_grid.max():
        raise ValueError(
            f"{concentration_ppm:.4f} ppmはLUT範囲外です。LUT範囲: "
            f"{concentration_grid.min():.4f}–{concentration_grid.max():.4f} ppm"
        )
    return np.array([
        np.interp(concentration_ppm, concentration_grid, sensor_lut[:, j])
        for j in range(sensor_lut.shape[1])
    ])


def build_enhancement_grid(background_ppm, max_enhancement_ppm, step_ppm, concentration_grid):
    max_allowed = concentration_grid.max() - background_ppm
    if background_ppm < concentration_grid.min():
        raise ValueError("背景メタン濃度がLUT下限未満です。")
    if max_enhancement_ppm > max_allowed + 1e-12:
        raise ValueError(
            f"背景濃度+最大増分がLUT上限を超えます。使用可能最大増分: {max_allowed:.4f} ppm"
        )
    grid = np.arange(0.0, max_enhancement_ppm + 0.5*step_ppm, step_ppm)
    grid[-1] = min(grid[-1], max_enhancement_ppm)
    return np.unique(np.append(grid, max_enhancement_ppm))


def make_enhancement_ratio_lut(sensor_lut, concentration_grid, background_ppm, enhancement_grid):
    bg = interpolate_absolute_lut_spectrum(background_ppm, concentration_grid, sensor_lut)
    bg = np.maximum(bg, 1e-30)
    ratio_lut = []
    for dc in enhancement_grid:
        enhanced = interpolate_absolute_lut_spectrum(background_ppm + dc, concentration_grid, sensor_lut)
        ratio_lut.append(enhanced / bg)
    return np.asarray(ratio_lut)

In [ ]:
enhancement_grid = build_enhancement_grid(
    BACKGROUND_CH4_PPM,
    MAX_ENHANCEMENT_PPM,
    ENHANCEMENT_STEP_PPM,
    absolute_concentration_grid
)

enhancement_ratio_lut = make_enhancement_ratio_lut(
    sensor_lut_absolute,
    absolute_concentration_grid,
    BACKGROUND_CH4_PPM,
    enhancement_grid
)

print("Background CH4 concentration:", BACKGROUND_CH4_PPM, "ppm")
print("Enhancement grid [ppm]:", enhancement_grid)
print("Enhancement-ratio LUT shape:", enhancement_ratio_lut.shape)

## 5. 合成メタンプルーム濃度増分マップ

In [ ]:
def make_advected_enhancement_plume(shape, peak_enhancement_ppm, center_yx=None,
                                     angle_deg=0.0, decay_length=18.0,
                                     cross_sigma=4.0, source_sigma=2.0):
    h, w = shape
    if center_yx is None:
        center_yx = (h//2, w//2)
    cy, cx = center_yx
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    theta = np.deg2rad(angle_deg)
    dx, dy = xx-cx, yy-cy
    along = dx*np.cos(theta) + dy*np.sin(theta)
    cross = -dx*np.sin(theta) + dy*np.cos(theta)
    downstream = np.maximum(along, 0.0)
    plume = np.exp(-downstream/max(decay_length,1e-6))*np.exp(-0.5*(cross/max(cross_sigma,1e-6))**2)
    plume *= along >= 0
    source = np.exp(-0.5*((dx/max(source_sigma,1e-6))**2 + (dy/max(source_sigma,1e-6))**2))
    plume = np.maximum(plume, source)
    if plume.max() > 0:
        plume /= plume.max()
    return peak_enhancement_ppm * plume


if PLUME_PEAK_ENHANCEMENT_PPM > enhancement_grid.max():
    raise ValueError("合成プルームのピーク増分が増分LUT上限を超えています。")

alpha_true_enhancement = make_advected_enhancement_plume(
    cube_background_true.shape[:2],
    PLUME_PEAK_ENHANCEMENT_PPM,
    PLUME_CENTER_YX,
    PLUME_ANGLE_DEG,
    PLUME_DECAY_PIX,
    PLUME_CROSS_SIGMA_PIX,
    PLUME_SOURCE_SIGMA_PIX
)
alpha_true_enhancement[~valid_mask] = np.nan

plt.figure(figsize=(6,5))
plt.imshow(alpha_true_enhancement)
plt.colorbar(label="Injected CH4 enhancement [ppm]")
plt.title("Synthetic CH4 enhancement plume")
plt.xlabel("x"); plt.ylabel("y"); plt.show()

## 6. HISUI背景画像へ濃度増分を注入

In [ ]:
def interpolate_ratio_cube(enhancement_map, enhancement_grid, ratio_lut):
    flat = np.asarray(enhancement_map, float).reshape(-1)
    finite = np.isfinite(flat)
    if np.any(finite & ((flat < enhancement_grid.min()) | (flat > enhancement_grid.max()))):
        raise ValueError("濃度増分マップに増分LUT範囲外の値があります。")
    out = np.full((flat.size, ratio_lut.shape[1]), np.nan)
    for j in range(ratio_lut.shape[1]):
        out[finite, j] = np.interp(flat[finite], enhancement_grid, ratio_lut[:, j])
    return out.reshape(enhancement_map.shape + (ratio_lut.shape[1],))


def inject_synthetic_methane_enhancement(cube_background, enhancement_map,
                                          enhancement_grid, ratio_lut, valid_mask):
    ratio_cube = interpolate_ratio_cube(enhancement_map, enhancement_grid, ratio_lut)
    injected = cube_background * ratio_cube
    injected[~valid_mask] = np.nan
    return injected, ratio_cube


cube_injected, ratio_cube = inject_synthetic_methane_enhancement(
    cube_background_true,
    alpha_true_enhancement,
    enhancement_grid,
    enhancement_ratio_lut,
    valid_mask
)
print("Injected cube shape:", cube_injected.shape)

In [ ]:
def plot_injection_spectrum(wavelengths, before, after, enhancement_map, y, x, xlim=None):
    plt.figure(figsize=(9,4))
    plt.plot(wavelengths, before[y,x], label="Before injection")
    plt.plot(wavelengths, after[y,x], label="After injection")
    plt.xlabel("Wavelength [nm]"); plt.ylabel("Radiance")
    plt.title(f"y={y}, x={x}, true enhancement={enhancement_map[y,x]:.3f} ppm")
    if xlim is not None: plt.xlim(*xlim)
    plt.grid(True); plt.legend(); plt.show()

peak_y, peak_x = np.unravel_index(np.nanargmax(alpha_true_enhancement), alpha_true_enhancement.shape)
plot_injection_spectrum(wavelengths, cube_background_true, cube_injected, alpha_true_enhancement, peak_y, peak_x, WINDOW_16)
plot_injection_spectrum(wavelengths, cube_background_true, cube_injected, alpha_true_enhancement, peak_y, peak_x, WINDOW_23)

## 7. 背景濃度付近のUASを作成

増分用UASは、背景濃度を基準とした放射輝度比から

$$
s(\lambda)=-\frac{\partial\ln T(\lambda,\Delta c)}{\partial\Delta c}
$$

として求める。単位は $\mathrm{ppm}^{-1}$ 

In [ ]:
def compute_uas_from_enhancement_ratio(enhancement_grid, ratio_lut, max_enhancement_ppm):
    use = (enhancement_grid >= 0) & (enhancement_grid <= max_enhancement_ppm)
    if use.sum() < 2:
        raise ValueError("UAS回帰に使える増分LUT点が2点未満です。")
    x = enhancement_grid[use]
    y = np.log(np.maximum(ratio_lut[use], 1e-30))
    design = np.column_stack([np.ones_like(x), x])
    coef, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
    return -coef[1], coef[0]

uas_all, uas_intercept = compute_uas_from_enhancement_ratio(
    enhancement_grid, enhancement_ratio_lut, UAS_MAX_ENHANCEMENT_PPM
)
mask_16 = (wavelengths >= WINDOW_16[0]) & (wavelengths <= WINDOW_16[1])
mask_23 = (wavelengths >= WINDOW_23[0]) & (wavelengths <= WINDOW_23[1])
if mask_16.sum() < 2 or mask_23.sum() < 2:
    raise ValueError(f"吸収窓内バンド数不足: 1.6 µm={mask_16.sum()}, 2.3 µm={mask_23.sum()}")

plt.figure(figsize=(9,4))
plt.plot(wavelengths[mask_16], uas_all[mask_16], label="1.6 µm UAS")
plt.plot(wavelengths[mask_23], uas_all[mask_23], label="2.3 µm UAS")
plt.xlabel("Wavelength [nm]"); plt.ylabel("UAS [ppm$^{-1}$]")
plt.grid(True); plt.legend(); plt.show()

## 8. Oracle背景を使った各吸収帯の線形MF

注入前cubeを真の画素別背景 $R_i$ とし、ターゲットを $t_i=-R_i\odot s$ とする。
ここでは簡易評価として共分散は対角近似。

In [ ]:
def estimate_band_variance(cube, band_mask, background_mask, floor=1e-12):
    x = cube[background_mask][:, band_mask]
    var = np.nanvar(x, axis=0, ddof=1)
    good = var[np.isfinite(var) & (var > floor)]
    replacement = np.median(good) if good.size else 1.0
    return np.where(np.isfinite(var) & (var > floor), var, replacement)


def oracle_linear_mf_map(observed_cube, background_cube, uas, band_mask, valid_mask, variance):
    obs = observed_cube[:,:,band_mask]
    bg = background_cube[:,:,band_mask]
    target = -bg * uas[band_mask][None,None,:]
    diff = obs-bg
    inv_var = 1.0/np.maximum(variance,1e-30)
    num = np.sum(diff*target*inv_var[None,None,:], axis=2)
    den = np.sum(target*target*inv_var[None,None,:], axis=2)
    out = np.full(valid_mask.shape, np.nan)
    safe = valid_mask & np.isfinite(den) & (den>1e-30)
    out[safe] = num[safe]/den[safe]
    return out

oracle_background_mask = valid_mask & (np.nan_to_num(alpha_true_enhancement, nan=0.0) < 1e-8)
variance_16 = estimate_band_variance(cube_background_true, mask_16, oracle_background_mask)
variance_23 = estimate_band_variance(cube_background_true, mask_23, oracle_background_mask)
alpha_mf_16 = oracle_linear_mf_map(cube_injected, cube_background_true, uas_all, mask_16, valid_mask, variance_16)
alpha_mf_23 = oracle_linear_mf_map(cube_injected, cube_background_true, uas_all, mask_23, valid_mask, variance_23)

## 9. 二吸収帯に共通の濃度増分を使うLUT探索

$$
\begin{aligned}
J_i(\Delta c)=&\sum_{\lambda\in W_{1.6}}
\frac{[L_i(\lambda)-R_i(\lambda)T(\lambda,\Delta c)]^2}{\sigma_\lambda^2}\\
&+\sum_{\lambda\in W_{2.3}}
\frac{[L_i(\lambda)-R_i(\lambda)T(\lambda,\Delta c)]^2}{\sigma_\lambda^2}
\end{aligned}
$$

In [ ]:
def dual_window_grid_retrieval(observed_cube, background_cube, ratio_lut, enhancement_candidates,
                               mask_16, mask_23, valid_mask, variance_16, variance_23):
    obs16, bg16 = observed_cube[:,:,mask_16], background_cube[:,:,mask_16]
    obs23, bg23 = observed_cube[:,:,mask_23], background_cube[:,:,mask_23]
    r16, r23 = ratio_lut[:,mask_16], ratio_lut[:,mask_23]
    iv16 = 1.0/np.maximum(variance_16,1e-30)
    iv23 = 1.0/np.maximum(variance_23,1e-30)
    best_cost = np.full(valid_mask.shape, np.inf)
    best_alpha = np.full(valid_mask.shape, np.nan)
    for k, dc in enumerate(enhancement_candidates):
        pred16 = bg16*r16[k][None,None,:]
        pred23 = bg23*r23[k][None,None,:]
        cost = np.sum((obs16-pred16)**2*iv16[None,None,:],axis=2) + \
               np.sum((obs23-pred23)**2*iv23[None,None,:],axis=2)
        improve = valid_mask & np.isfinite(cost) & (cost<best_cost)
        best_cost[improve] = cost[improve]
        best_alpha[improve] = dc
    best_cost[~valid_mask] = np.nan
    return best_alpha, best_cost

alpha_dual_grid, dual_grid_cost = dual_window_grid_retrieval(
    cube_injected, cube_background_true, enhancement_ratio_lut, enhancement_grid,
    mask_16, mask_23, valid_mask, variance_16, variance_23
)

## 10. 連続濃度増分への非線形精密化

In [ ]:
def interpolate_ratio_vector(dc, enhancement_grid, ratio_lut):
    dc = float(np.clip(dc, enhancement_grid.min(), enhancement_grid.max()))
    return np.array([np.interp(dc, enhancement_grid, ratio_lut[:,j]) for j in range(ratio_lut.shape[1])])


def refine_dual_window_pixel(observed, background, initial_dc, enhancement_grid, ratio_lut,
                             mask_16, mask_23, variance_16, variance_23):
    sw16 = 1.0/np.sqrt(np.maximum(variance_16,1e-30))
    sw23 = 1.0/np.sqrt(np.maximum(variance_23,1e-30))
    def residual(p):
        ratio = interpolate_ratio_vector(p[0], enhancement_grid, ratio_lut)
        pred = background*ratio
        return np.concatenate([(observed[mask_16]-pred[mask_16])*sw16,
                               (observed[mask_23]-pred[mask_23])*sw23])
    result = least_squares(
        residual,
        x0=np.array([initial_dc],float),
        bounds=(enhancement_grid.min(), enhancement_grid.max()),
        method="trf"
    )
    return float(result.x[0]), float(np.sum(result.fun**2)), bool(result.success)


def refine_dual_window_map(observed_cube, background_cube, initial_map, valid_mask,
                           enhancement_grid, ratio_lut, mask_16, mask_23,
                           variance_16, variance_23):
    refined = initial_map.copy()
    cost_map = np.full(valid_mask.shape, np.nan)
    success_map = np.zeros(valid_mask.shape, bool)
    indices = np.argwhere(valid_mask)
    for n,(y,x) in enumerate(indices, start=1):
        initial = initial_map[y,x] if np.isfinite(initial_map[y,x]) else 0.0
        refined[y,x], cost_map[y,x], success_map[y,x] = refine_dual_window_pixel(
            observed_cube[y,x], background_cube[y,x], initial,
            enhancement_grid, ratio_lut, mask_16, mask_23, variance_16, variance_23
        )
        if n % 1000 == 0:
            print(f"Refined {n}/{len(indices)} pixels")
    refined[~valid_mask] = np.nan
    return refined, cost_map, success_map

alpha_dual_refined, dual_refined_cost, dual_refined_success = refine_dual_window_map(
    cube_injected, cube_background_true, alpha_dual_grid, valid_mask,
    enhancement_grid, enhancement_ratio_lut, mask_16, mask_23, variance_16, variance_23
)

## 11. 二吸収間の不整合スコア

In [ ]:
def robust_std(image, mask):
    values = image[mask]
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    med = np.median(values)
    mad = np.median(np.abs(values-med))
    return 1.4826*mad if mad>0 else np.std(values)

std16 = robust_std(alpha_mf_16, oracle_background_mask)
std23 = robust_std(alpha_mf_23, oracle_background_mask)
inter_window_inconsistency = np.abs(alpha_mf_16-alpha_mf_23)/(std16+std23+1e-12)
inter_window_inconsistency[~valid_mask] = np.nan

## 12. 回帰・検出評価

In [ ]:
def regression_metrics(true_map, estimated_map, valid_mask):
    use = valid_mask & np.isfinite(true_map) & np.isfinite(estimated_map)
    true = true_map[use]; est = estimated_map[use]
    residual = est-true
    if true.size>=2 and np.var(true)>0:
        slope, intercept = np.polyfit(true, est, 1)
        r2 = np.corrcoef(true, est)[0,1]**2
    else:
        slope=intercept=r2=np.nan
    return {
        "n": int(true.size),
        "bias_ppm": float(np.mean(residual)),
        "mae_ppm": float(np.mean(np.abs(residual))),
        "rmse_ppm": float(np.sqrt(np.mean(residual**2))),
        "slope": slope,
        "intercept_ppm": intercept,
        "r2": r2
    }


def detection_metrics(true_map, estimated_map, valid_mask, background_mask, true_threshold_ppm=0.05):
    threshold = np.nanmedian(estimated_map[background_mask]) + 3.0*robust_std(estimated_map, background_mask)
    true_plume = valid_mask & (true_map>=true_threshold_ppm)
    detected = valid_mask & (estimated_map>=threshold)
    tp = int(np.sum(true_plume & detected))
    fp = int(np.sum((~true_plume) & detected & valid_mask))
    fn = int(np.sum(true_plume & (~detected)))
    precision = tp/(tp+fp) if tp+fp else np.nan
    recall = tp/(tp+fn) if tp+fn else np.nan
    f1 = 2*precision*recall/(precision+recall) if np.isfinite(precision) and np.isfinite(recall) and precision+recall>0 else np.nan
    return {"threshold_ppm":threshold,"precision":precision,"recall":recall,"f1":f1,"tp":tp,"fp":fp,"fn":fn}

methods = {
    "Linear MF, 1.6 µm": alpha_mf_16,
    "Linear MF, 2.3 µm": alpha_mf_23,
    "Dual-window LUT grid": alpha_dual_grid,
    "Dual-window nonlinear refinement": alpha_dual_refined
}

rows=[]
for name, estimate in methods.items():
    rows.append({
        "method":name,
        **regression_metrics(alpha_true_enhancement, estimate, valid_mask),
        **detection_metrics(alpha_true_enhancement, estimate, valid_mask, oracle_background_mask)
    })
summary_df = pd.DataFrame(rows)
display(summary_df)

## 13. マップと真値ー推定値散布図

In [ ]:
def show_map(image, title, label="CH4 enhancement [ppm]", vmin=None, vmax=None):
    plt.figure(figsize=(6,5))
    plt.imshow(image, vmin=vmin, vmax=vmax)
    plt.colorbar(label=label)
    plt.title(title); plt.xlabel("x"); plt.ylabel("y"); plt.show()

common_max = np.nanpercentile(alpha_true_enhancement[valid_mask], 99.5)
show_map(alpha_true_enhancement, "True synthetic enhancement", vmin=0, vmax=common_max)
show_map(alpha_mf_16, "Linear MF: 1.6 µm")
show_map(alpha_mf_23, "Linear MF: 2.3 µm")
show_map(alpha_dual_grid, "Dual-window LUT grid", vmin=0, vmax=common_max)
show_map(alpha_dual_refined, "Dual-window nonlinear refinement", vmin=0, vmax=common_max)
show_map(inter_window_inconsistency, "Inter-window inconsistency", label="Normalized inconsistency")

for name, estimate in methods.items():
    use = valid_mask & np.isfinite(alpha_true_enhancement) & np.isfinite(estimate)
    true = alpha_true_enhancement[use]; est = estimate[use]
    limit = max(np.nanmax(true), np.nanmax(est), 1e-6)
    plt.figure(figsize=(5,5))
    plt.scatter(true, est, s=8, alpha=0.4)
    plt.plot([0,limit],[0,limit],"--")
    plt.xlabel("True enhancement [ppm]")
    plt.ylabel("Retrieved enhancement [ppm]")
    plt.title(name); plt.grid(True); plt.show()